## 🧠 Model Architecture

This project uses a lightweight Convolutional Neural Network (CNN) for multi-class galaxy morphology classification.

The model predicts one of three classes:
- **Elliptical**
- **Spiral**
- **Artifact**

---

### 🔧 Input

- Image size: **128 × 128 × 3 (RGB)**  
- Pixel values normalized to `[0, 1]`

---

### 🔄 Data Augmentation

Applied on-the-fly during training:

- Random horizontal flip  
- Random rotation (~±27°)  
- Random zoom (±10%)  
- Random contrast adjustment  

**Purpose:** improve generalization and reduce overfitting.

---

### 🧱 Feature Extraction (Convolutional Blocks)

#### Block 1

Conv2D(32, 3×3, padding=“same”, no bias)
→ BatchNormalization
→ ReLU
→ MaxPooling2D

- Learns basic features: edges, gradients, simple textures  
- Reduces spatial resolution: `128×128 → 64×64`

---

#### Block 2

Conv2D(64, 3×3, padding=“same”, no bias)
→ BatchNormalization
→ ReLU
→ MaxPooling2D

- Learns intermediate structures: shapes, brightness patterns  
- Captures early morphological features  
- Resolution: `64×64 → 32×32`

---

#### Block 3

Conv2D(128, 3×3, padding=“same”, no bias)
→ BatchNormalization
→ ReLU

- Learns higher-level galaxy features:
  - smooth light profiles  
  - disk structures  
  - asymmetries  
  - spiral patterns  

---

### 🔻 Feature Compression

GlobalAveragePooling2D

- Converts feature maps into a compact vector  
- Reduces parameters vs. Flatten  
- Focuses on **presence of features**, not exact position  

---

### 🧩 Classification Head

Dropout(0.35)
→ Dense(64, ReLU)
→ Dropout(0.25)
→ Dense(3, Softmax)

- Combines extracted features  
- Applies regularization (Dropout)  
- Outputs class probabilities  

---

### 📤 Output

[ P(Elliptical), P(Spiral), P(Artifact) ]

Probabilities sum to 1 (Softmax).

---

### ⚙️ Training Setup

- **Optimizer:** Adam (learning rate = 1e-3)  
- **Loss:** sparse categorical crossentropy  
- **Metric:** accuracy  

#### Callbacks:
- EarlyStopping (patience = 3)  
- ReduceLROnPlateau (adaptive learning rate)

---

### 🧭 Architectural Summary

Input (128×128×3)
→ Augmentation
→ Conv(32) + BN + ReLU + MaxPool
→ Conv(64) + BN + ReLU + MaxPool
→ Conv(128) + BN + ReLU
→ GlobalAveragePooling
→ Dropout
→ Dense(64, ReLU)
→ Dropout
→ Dense(3, Softmax)

---

### 🧪 Design Rationale

- **Compact CNN** — efficient and easy to train  
- **BatchNorm + ReLU** — stable gradients and faster convergence  
- **GlobalAveragePooling** — fewer parameters, better generalization  
- **Dropout** — regularization against overfitting  
- **Data augmentation** — robustness to observational variation  

---

### ⚠️ Limitations

- No transfer learning (e.g., ResNet / EfficientNet)  
- Labels derived from crowd vote fractions (noisy)  
- No class imbalance handling  
- Limited depth → may miss fine morphological details  

---

### 🚀 Future Improvements

- Transfer learning with pretrained CNNs  
- Class weighting or focal loss  
- Confidence-weighted labels  
- Larger input resolution  
- Advanced augmentations (CutMix, MixUp)

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
from pathlib import Path

from gzml.bootstrap import get_paths

tf.random.set_seed(42)
np.random.seed(42)

# -----------------------------------------------------------------------------
# Paths (same logic as in .py scripts)
# -----------------------------------------------------------------------------
paths = get_paths()  # repo root discovery + ensure dirs

OUT_DIR = paths.outputs                 # <repo>/outputs
FIG_DIR = paths.figures                 # <repo>/outputs/figures
MODEL_DIR = paths.outputs / "models"    # keep consistent with train_gz2.py
PRED_DIR = paths.outputs / "predictions"  # optional, for future inference artifacts

FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "gz2_cnn.keras"
TRAIN_CURVES_PNG = FIG_DIR / "gz2_training_curves.png"
PRED_GRID_PNG = FIG_DIR / "gz2_prediction_grid.png"

print("[Paths]")
print(" - outputs:", OUT_DIR)
print(" - figures:", FIG_DIR)
print(" - models :", MODEL_DIR)
print(" - model  :", MODEL_PATH)

# === Parameters ===
IMG_SIZE = (128, 128)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS = 10

# === Preprocessing ===
def preprocess(example):
    image = tf.image.resize(example["image"], IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    table = example["table1"]
    votes = [
        table["t01_smooth_or_features_a01_smooth_fraction"],
        table["t01_smooth_or_features_a02_features_or_disk_fraction"],
        table["t01_smooth_or_features_a03_star_or_artifact_fraction"],
    ]
    label = tf.argmax(tf.stack(votes), axis=0)
    return image, label

# === Загрузка датасета ===
print("🔽 Loading Galaxy Zoo 2...")
ds_full = tfds.load("galaxy_zoo2", split="train", shuffle_files=True)

print("📦 Counting elements...")
n_total = 0
for _ in tqdm(ds_full, desc="Counting records"):
    n_total += 1

train_size = int(n_total * 0.8)
test_size = n_total - train_size
print(f"🔢 TOTAL: {n_total} | Training suite: {train_size} | Test suite: {test_size}")

# === Разделение на train/test ===
ds_train_raw = ds_full.take(train_size)
ds_test_raw  = ds_full.skip(train_size)

def prepare(ds):
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

ds_train = prepare(ds_train_raw)
ds_test  = prepare(ds_test_raw)

data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.15),     # ~±27°
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.15),
    ],
    name="augmentation",
)

# === Model ===
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),

    data_augmentation,  # <--- добавили

    tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),

    tf.keras.layers.GlobalAveragePooling2D(),   # <--- вместо Flatten()
    tf.keras.layers.Dropout(0.35),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Dense(3, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
    ),
]

# === Training ===
print("🚀 Training model...")

history = model.fit(
    ds_train,
    validation_data=ds_test,
    epochs=EPOCHS,
    callbacks=callbacks,
)

# === Teting ===
print("📊 Testing sample:")
test_loss, test_acc = model.evaluate(ds_test)
print(f"Test Accuracy: {test_acc:.4f}")

# === Графики (сохранить как в скрипте) ===
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(TRAIN_CURVES_PNG, dpi=150)
plt.show()  # оставить для просмотра в ноутбуке
plt.close()

# === Prediction examples (grid + save) ===
class_names = ["Elliptical", "Spiral", "Artifact"]
for images, labels in ds_test.take(1):
    preds = model.predict(images, verbose=0)
    preds_labels = np.argmax(preds, axis=1)

    plt.figure(figsize=(10, 10))
    for i in range(9):
        plt.subplot(3, 3, i+1)
        plt.imshow(images[i])
        plt.title(f"Pred: {class_names[preds_labels[i]]}\nTrue: {class_names[int(labels[i])]}")
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(PRED_GRID_PNG, dpi=150)
    plt.show()
    plt.close()
    break

# === Сохранение модели (как в скрипте) ===
model.save(MODEL_PATH)
print("💾 Model saved:", MODEL_PATH)
print("🖼️ Figures saved:", TRAIN_CURVES_PNG, "and", PRED_GRID_PNG)

[Paths]
 - outputs: /Users/mloktionov/PycharmProjects/Stellar_Attractor/astrolab/astrolab/GalaxyZooML/outputs
 - figures: /Users/mloktionov/PycharmProjects/Stellar_Attractor/astrolab/astrolab/GalaxyZooML/outputs/figures
 - models : /Users/mloktionov/PycharmProjects/Stellar_Attractor/astrolab/astrolab/GalaxyZooML/outputs/models
 - model  : /Users/mloktionov/PycharmProjects/Stellar_Attractor/astrolab/astrolab/GalaxyZooML/outputs/models/gz2_cnn.keras
🔽 Loading Galaxy Zoo 2...
📦 Counting elements...


Counting records: 100%|██████████| 239573/239573 [03:05<00:00, 1291.15it/s]


🔢 TOTAL: 239573 | Training suite: 191658 | Test suite: 47915
🚀 Training model...
Epoch 1/10
 292/2995 ━━━━━━━━━━━━━━━━━━━━ 17:56 398ms/step - accuracy: 0.7172 - loss: 0.5881

KeyboardInterrupt: 

In [ ]:
# 📦 Imports
import csv
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from gzml.bootstrap import get_paths

# -----------------------------------------------------------------------------
# Paths (single source of truth)
# -----------------------------------------------------------------------------
paths = get_paths()

MODEL_PATH = paths.outputs / "models" / "gz2_cnn.keras"
IMAGES_DIR = paths.galaxies / "Sample_1"                # <repo>/galaxy images
OUT_DIR    = paths.outputs / "predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUT_DIR / "predictions_model_1_sample_1.csv"

print("[Paths]")
print(" - model      :", MODEL_PATH)
print(" - images dir :", IMAGES_DIR)
print(" - output csv :", OUTPUT_CSV)

# -----------------------------------------------------------------------------
# Parameters (MUST match training)
# -----------------------------------------------------------------------------
IMG_SIZE = (128, 128)
CLASS_NAMES = ["Elliptical", "Spiral", "Artifact"]

# -----------------------------------------------------------------------------
# Load model
# -----------------------------------------------------------------------------
print("🧠 Loading model...")
model = load_model(MODEL_PATH)

# -----------------------------------------------------------------------------
# Inference (with confidence & margin)
# -----------------------------------------------------------------------------
results: list[tuple[str, str, float, float]] = []

CONF_THRESHOLD = 0.60
MARGIN_THRESHOLD = 0.15

print("🔍 Classifying images...")
for path in tqdm(sorted(IMAGES_DIR.iterdir()), desc="Classification"):
    if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue

    try:
        img = Image.open(path).convert("RGB").resize(IMG_SIZE)
        img_array = np.asarray(img, dtype=np.float32) / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        preds = model.predict(img_array, verbose=0)[0]

        # top-1 / top-2
        order = np.argsort(preds)[::-1]
        p1, p2 = preds[order[0]], preds[order[1]]
        margin = float(p1 - p2)
        confidence = float(p1)

        label = CLASS_NAMES[order[0]]

        if confidence < CONF_THRESHOLD or margin < MARGIN_THRESHOLD:
            label = "Unknown"

        results.append((path.name, label, confidence, margin))

    except Exception as e:
        print(f"⚠️ Failed on {path.name}: {e}")

# -----------------------------------------------------------------------------
# Save CSV
# -----------------------------------------------------------------------------
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "filename",
        "predicted_class",
        "confidence",
        "margin",
    ])
    writer.writerows(results)

print("✅ Classification finished")
print("📄 Results saved to:", OUTPUT_CSV.resolve())

[Paths]
 - model      : /Users/mloktionov/PycharmProjects/Stellar_Attractor/astrolab/astrolab/GalaxyZooML/outputs/models/gz2_cnn.keras
 - images dir : /Users/mloktionov/PycharmProjects/Stellar_Attractor/astrolab/astrolab/GalaxyZooML/Samples/Sample_1
 - output csv : /Users/mloktionov/PycharmProjects/Stellar_Attractor/astrolab/astrolab/GalaxyZooML/outputs/predictions/predictions_model_1_sample_1.csv
🧠 Loading model...
🔍 Classifying images...


Classification: 100%|██████████| 11/11 [00:00<00:00, 17.91it/s]

✅ Classification finished
📄 Results saved to: /Users/mloktionov/PycharmProjects/Stellar_Attractor/astrolab/astrolab/GalaxyZooML/outputs/predictions/predictions_model_1_sample_1.csv


In [ ]:
# -----------------------------------------------------------------------------
# Optional: accuracy if ground truth exists
# -----------------------------------------------------------------------------
from pathlib import Path
import pandas as pd

LABELS_CSV = IMAGES_DIR / "labels.csv"
df_true = pd.read_csv(LABELS_CSV)

if LABELS_CSV.exists():
    from sklearn.metrics import accuracy_score

    print("📊 Ground truth found, computing accuracy...")

    df_true = pd.read_csv(LABELS_CSV)
    true_map = {Path(fn).stem.strip(): str(tc).strip() for fn, tc in zip(df_true["filename"], df_true["true_class"])}

    y_true, y_pred = [], []
    for fname, pred, conf, margin in results:   # если results теперь 4-колоночный
        key = Path(fname).stem.strip()
        if key in true_map:
            y_true.append(true_map[key])
            y_pred.append(str(pred).strip())

    for fname, pred, *_ in results:
        if fname in true_map:
            y_true.append(true_map[fname])
            y_pred.append(pred)

    if y_true:
        acc = accuracy_score(y_true, y_pred)
        print(f"✅ Accuracy on labeled subset: {acc:.4f}")
    else:
        print("⚠️ No matching labels found for evaluated images.")
else:
    print("ℹ️ No labels.csv found — skipping accuracy computation.")

📊 Ground truth found, computing accuracy...
✅ Accuracy on labeled subset: 0.8000
